# Main Figure 4 - Lung fibrosis DST-GNN

            This notebook is a cleaned, English-commented manuscript code file.

            **Provenance.** `Vannan_2023_Lung_Fibrosis/notebook/GNN modelling.ipynb` and A100 `/data/taobo.hu/tmp/dst_gnn_release_repo`.

            The notebook keeps the original manuscript data paths where they were found. If a path points to
            `/data/taobo.hu` or `/mnt/taobo.hu`, run the notebook on the A100 server or adapt the path to the
            matching mounted Y-drive location.


In [ ]:
# Panel mapping:
# - Figure 4a-d, shared setup. Imports PyTorch/PyG, fixes random seeds, and defines the stage labels used by the DST-GNN analysis.
#

from pathlib import Path
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

INPUT_CSV = Path("Y:/long/publication_datasets/Vannan_2023_Lung_Fibrosis/notebook/cophenetic_distances_searcher_D_score_in_all_samples.csv")
OUTPUT_DIR = Path("outputs/main_figure_4_dst_gnn")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
STAGE_MAP = {"HD": "T1", "LA": "T2", "MA": "T3"}
STAGE_ORDER = ["T1", "T2", "T3"]
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)


In [ ]:
# Panel mapping:
# - Figure 4a. Converts sample-level SSS edges into one directed disease-stage graph per T1/T2/T3 stage.
#

def load_stage_matrices(csv_path):
    """Average sample-level COSTE/SSS edges into one directed graph per disease stage."""
    df = pd.read_csv(csv_path)
    df["stage"] = df["group"].map(STAGE_MAP).fillna(df["group"])
    nodes = sorted(set(df["row"]).union(df["column"]))
    matrices = {}
    for stage in STAGE_ORDER:
        matrix = pd.DataFrame(1.0, index=nodes, columns=nodes, dtype=float)
        avg = df[df["stage"].eq(stage)].groupby(["row", "column"], as_index=False)["value"].mean()
        for _, record in avg.iterrows():
            matrix.loc[record["row"], record["column"]] = float(record["value"])
        np.fill_diagonal(matrix.values, 0.0)
        matrices[stage] = matrix
        matrix.to_csv(OUTPUT_DIR / f"observed_{stage}_sss.csv")
    return matrices, nodes

stage_matrices, node_names = load_stage_matrices(INPUT_CSV)


In [ ]:
# Panel mapping:
# - Figure 4b. Defines graph edges and the compact DST-GNN model architecture used to learn stage-specific spatial transitions.
#

def matrix_to_edges(matrix):
    values = matrix.to_numpy(dtype=np.float32)
    src, dst = np.nonzero(np.ones_like(values, dtype=bool))
    edge_index = torch.tensor(np.vstack([src, dst]), dtype=torch.long)
    edge_weight = torch.tensor(1.0 - values[src, dst], dtype=torch.float32).clamp(min=0.0, max=1.0)
    return edge_index, edge_weight


class DSTGNN(torch.nn.Module):
    """Compact manuscript-aligned diffusion temporal graph model."""
    def __init__(self, node_count, hidden_channels=32):
        super().__init__()
        self.register_buffer("node_features", torch.eye(node_count))
        self.conv1 = GCNConv(node_count, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.gru = torch.nn.GRUCell(hidden_channels, hidden_channels)
        self.decoder = torch.nn.Sequential(torch.nn.Linear(hidden_channels * 2, hidden_channels), torch.nn.ReLU(), torch.nn.Linear(hidden_channels, 1), torch.nn.Sigmoid())

    def encode_stage(self, edge_index, edge_weight):
        h = F.relu(self.conv1(self.node_features, edge_index, edge_weight))
        return F.relu(self.conv2(h, edge_index, edge_weight))

    def decode_matrix(self, h):
        n = h.shape[0]
        left = h[:, None, :].expand(n, n, h.shape[1])
        right = h[None, :, :].expand(n, n, h.shape[1])
        return self.decoder(torch.cat([left, right], dim=-1)).squeeze(-1)


In [ ]:
# Panel mapping:
# - Figure 4c-d. Trains the DST-GNN model, saves training history, and exports dynamic node and edge-change scores for the final panels.
#

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DSTGNN(len(node_names), hidden_channels=32).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
edges = {stage: tuple(t.to(device) for t in matrix_to_edges(matrix)) for stage, matrix in stage_matrices.items()}
targets = {stage: torch.tensor(1.0 - matrix.to_numpy(dtype=np.float32), device=device) for stage, matrix in stage_matrices.items()}

history = []
for epoch in range(400):
    optimizer.zero_grad()
    state = torch.zeros((len(node_names), 32), device=device)
    loss = torch.tensor(0.0, device=device)
    for current_stage, next_stage in zip(STAGE_ORDER[:-1], STAGE_ORDER[1:]):
        encoded = model.encode_stage(*edges[current_stage])
        state = model.gru(encoded, state)
        loss = loss + F.mse_loss(model.decode_matrix(state), targets[next_stage])
    loss.backward(); optimizer.step()
    history.append({"epoch": epoch + 1, "loss": float(loss.detach().cpu())})
pd.DataFrame(history).to_csv(OUTPUT_DIR / "training_history.csv", index=False)

delta = stage_matrices["T3"] - stage_matrices["T1"]
delta.abs().mean(axis=1).sort_values(ascending=False).rename("drift_score").to_csv(OUTPUT_DIR / "top_dynamic_nodes.csv")
edge_changes = delta.stack().rename("delta_sss").reset_index().rename(columns={"level_0": "source", "level_1": "target"})
edge_changes["abs_delta_sss"] = edge_changes["delta_sss"].abs()
edge_changes.sort_values("abs_delta_sss", ascending=False).to_csv(OUTPUT_DIR / "top_edge_changes.csv", index=False)
